# Attention and Transformers


Figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*
(CC-BY-NC-ND, © MIT Press), unless another source is credited on the slide.

Code cells adapted from the UDL notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).


## A brief history of transformers

![A brief history of transformers. Figure: © SuperDataScience; chronological analysis inspired by Andrej Karpathy's lecture.](figs/10-Attention-and-Transformers/transformer-history-timeline.png)


## A Neural Probabilistic Language Model

*Bengio et al., 2000 and 2003*

![Neural architecture: $f(i, w_{t-1}, \cdots, w_{t-n+1}) = g(i, C(w_{t-1}), \cdots, C(w_{t-n+1}))$ where $g$ is the neural network and $C(i)$ is the $i$-th word feature vector.](figs/10-Attention-and-Transformers/nplm-architecture.png) to w_{t-1} look up feature vectors C(w) in a shared matrix C; these feed a tanh hidden layer, then a softmax layer whose i-th output is P(w_t = i | context). Dashed arrows show optional direct connections from the feature vectors to the softmax." width="100%"}

- Softmax over the entire vocabulary
- Single FC hidden layer (50-100 hidden units) w/ tanh activation
- Optional direct connections
- Feature (embedding) vectors, $C(w_t)$
- $C$ is a $|V| \times m$ matrix, $m = 30, 60$ or $100$
- $w_t \in V$ words in the vocabulary ~16k-18k


- Build a probabilistic language model from NNs
- Feed forward network with shared learned parameters, C, that create embeddings
- Precursor to Word2Vec
- Predicts the probability of a word at time $t$, based on the context of the last $n$ words
- 13-14M parameters

**Limited to context length of $n \in \{3, 4, 5, 6\}$. Tiny!!**


Y. Bengio, R. Ducharme, P. Vincent, and C. Jauvin, "A Neural Probabilistic Language Model," *Journal of Machine Learning Research*, vol. 3, pp. 1137–1155, Feb. 2003.

## Sequence to Sequence Learning with Neural Networks

*Sutskever et al. (2014)*

![An LSTM cell. **Use LSTMs.**](figs/10-Attention-and-Transformers/lstm-cell.png)

![The encoder reads the English sentence and the decoder produces the French one; the single link between them is the bottleneck.](figs/10-Attention-and-Transformers/seq2seq-encoder-decoder.png)


- Used 4-layer LSTMs in an Encoder/Decoder structure
- Used 1000-D embeddings learned in training w/ 160K source vocabulary
- Estimate the probability of $p(y_1, \ldots, y_{T'} \mid x_1, \ldots, x_T)$ where $T'$ may differ from $T$, and $T, T'$ were ~20-30
- Encoder mapped sequence to a fixed size token (1000-D hidden state)
- The hidden state may not encode all the information needed by the decoder
- ~380M parameters

***Bottleneck* between Encoder and Decoder!**


I. Sutskever, O. Vinyals, and Q. V. Le, "[Sequence to Sequence Learning with Neural Networks](https://proceedings.neurips.cc/paper/2014/hash/a14ac55a4f27472c5d894ec1c3c743d2-Abstract.html)," in *Advances in Neural Information Processing Systems*, Curran Associates, Inc., 2014.

## Neural Machine Translation by Jointly Learning to Align and Translate

*Bahdanau, Cho & Bengio (2014-15)*

![**Add Attention.** Annotations, $h_j$, weighted by attention, $\alpha_{t,j}$, and summed. $T = 30$ or $50$.](figs/10-Attention-and-Transformers/bahdanau-attention-rnn.png) to alpha_{t,T} and summed into the decoder state s_t that produces y_t." width="62%"}


- Used a bidirectional RNN with gated (GRU-style) hidden units ([Cho et al., 2014](https://arxiv.org/abs/1406.1078)), not LSTMs
- Concatenated forward and backward hidden states called "annotations"
- Automatically "soft-search" parts of input that influence the output
- Overcomes the bottleneck of a fixed size hidden state between encoder and decoder
- Significantly improved ability to comprehend longer sequences


## Neural Machine Translation by Jointly Learning to Align and Translate (cont.)

![Four sample alignments found by RNNsearch-50. The x-axis and y-axis of each plot correspond to the words in the source sentence (English) and the generated translation (French), respectively. Each pixel shows the weight $\alpha_{ij}$ of the annotation of the $j$-th source word for the $i$-th target word, in grayscale (0: black, 1: white). (a) an arbitrary sentence. (b–d) three randomly selected samples among the sentences without any unknown words and of length between 10 and 20 words from the test set. Figure: [Bahdanau et al.](http://arxiv.org/abs/1409.0473), Fig. 3.](figs/10-Attention-and-Transformers/bahdanau-alignments.png)

## Attention is All You Need

*Vaswani et al. (2017)*

![**Remove LSTMs.**](figs/10-Attention-and-Transformers/transformer-architecture.png)


- Removed LSTMs and didn't use convolutions
- Only attention mechanisms and MLPs
- Parallelizable by removing sequential hidden state computation
- Outperformed all previous models


A. Vaswani et al., "[Attention is All you Need](https://arxiv.org/abs/1706.03762)," NIPS 2017.

## Transformers applied to many NLP applications

- Translation
- Question answering
- Summarizing
- Generating new text
- Correcting spelling and grammar
- Finding entities
- Classifying bodies of text
- Changing style etc.

## Topics

- Motivation
- Dot-product self-attention
- Applying Self-Attention
- Position Encoding: sinusoids, then RoPE
- The Transformer Architecture
- Three Types of NLP Transformer Models

# Motivation

Before defining self-attention we look at what a network for text has to do:
handle long inputs of varying length, and connect words to one another in a way
that depends on the words themselves.


## Motivation

Design neural network to encode and process text:

> [The restaurant refused to serve me a ham sandwich, because it only cooks vegetarian food. In the end, they just gave me two slices of bread. Their ambience was just as good as the food and service.]{style="color: #e8890c;"}

- Create a vocabulary of words (or word parts)
- Encode to a $D$-dimensional embedding vector.
- We'll look at tokenization and embedding encoding later.
- For now, assume a word is a token.


![](figs/10-Attention-and-Transformers/word-embedding-vectors.png)

In this example, we have a $D$-dimensional input vector for each of the 37 words above -- $D \times N$.

Normally we would represent punctuation, capitalization, spaces, etc. as well.


## Standard fully-connected layer

$$
\mathbf{h} = \mathbf{a}[\boldsymbol{\beta} + \boldsymbol{\Omega}\mathbf{x}]
$$

![Assuming $D$ inputs and $D$ hidden units.](figs/10-Attention-and-Transformers/fc-layer-network.png)


Problem:

- token (word) vectors may be 512 or 1024 dimensional
- need to process large segment of text
- Hence, would require a very large number of parameters
- Can't cope with text of different lengths

Conclusion:

- We need a model where parameters don't increase with input length


## Motivation (cont.)

Design neural network to encode and process text:

> [The **restaurant** refused to serve me a ham sandwich, because it only cooks vegetarian food. In the end, they just gave me two slices of bread. **Their** ambience was just as good as the food and service.]{style="color: #e8890c;"}

The word [their]{style="color: #e8890c;"} must "attend to" the word [restaurant]{style="color: #e8890c;"}.

Conclusions:

- There must be connections between the words.
- The strength of these connections will depend on the words themselves.


So we need to efficiently process long strings of text *and* relate words across long context lengths. **Self-attention addresses both problems.**


# Dot-product self-attention

Self-attention maps $N$ input vectors to $N$ output vectors. Each output is a
weighted sum of *values* computed from every input, with weights that depend on
the inputs themselves.


## Dot-Product Self-Attention

1. Shares parameters to cope with long input passages of different lengths
2. Contains connections between word representations that depend on the words themselves
3. Takes $N$ inputs, $\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_N$, each of dimension $D \times 1$ and returns $N$ *outputs* of size $D \times 1$
4. In NLP, each input is a word or word fragment
    1. We'll talk about tokenization later

## Dot-product self attention -- Compute *values*

First, a set of *values* are computed for each input:

$$
\mathbf{v}_n = \boldsymbol{\beta}_v + \boldsymbol{\Omega}_v \mathbf{x}_n
$$

where $\boldsymbol{\beta}_v \in \mathbb{R}^{D \times 1}$ and $\boldsymbol{\Omega}_v \in \mathbb{R}^{D \times D}$ are biases and weights

## Dot-product self attention

- Then, the $n^{th}$ output is a weighted sum of all values $\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_N$:

$$
\mathbf{sa}_n[\mathbf{x}_1, \ldots, \mathbf{x}_N] = \sum_{m=1}^{N} \mathrm{a}[\mathbf{x}_m, \mathbf{x}_n] \mathbf{v}_m
$$

- $\mathbf{sa}_n[\cdot]$ is the $n^{th}$ *output* of self-attention for the sequence $\mathbf{x}_1, \ldots, \mathbf{x}_N$.
- $\mathrm{a}[\mathbf{x}_m, \mathbf{x}_n]$ are *scalar* self-attention weights that represent how much attention the $n^{th}$ token should pay to the $m^{th}$ token
- $\mathrm{a}[\cdot, \mathbf{x}_n]$ are non-negative and sum to one
- Weights depend on the inputs themselves

## Attention as routing


![Attention as routing: each output $\mathbf{sa}[\mathbf{x}_n]$ is a weighted sum of the values of all inputs.](figs/10-Attention-and-Transformers/attention-routing-3.png)


- Inputs → Values: linear transform. The weights into each output sum to 1.
- Here: # of inputs, $N = 3$; dimension of each input, $D = 4$
- We'll show how to calculate the self-attention weights shortly.

## *Values* computation scales linearly with number of inputs

![](figs/10-Attention-and-Transformers/values-linear-transform.png)


- Same weights and biases for every input
- $\boldsymbol{\beta}_v \in \mathbb{R}^{D \times 1}$ and $\boldsymbol{\Omega}_v \in \mathbb{R}^{D \times D}$


## And we can view as a sparse matrix operation

![](figs/10-Attention-and-Transformers/values-sparse-matrix.png)


- Repeating same weights


# Applying Self-Attention

Next we compute the attention weights $\mathrm{a}[\mathbf{x}_m, \mathbf{x}_n]$
themselves, from *queries* and *keys* computed from the inputs.


## Attention weights

- Compute $N$ "queries" and $N$ "keys" from input

$$
\begin{aligned}
\mathbf{q}_n &= \boldsymbol{\beta}_q + \boldsymbol{\Omega}_q \mathbf{x}_n \\
\mathbf{k}_n &= \boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{x}_n,
\end{aligned}
$$

- Take dot products and pass through softmax:

$$
\begin{aligned}
\mathrm{a}[\mathbf{x}_m, \mathbf{x}_n] &= \mathrm{softmax}_m\left[\mathbf{k}_m^T \mathbf{q}_n\right] \\
&= \frac{\exp\left[\mathbf{k}_m^T \mathbf{q}_n\right]}{\sum_{m'=1}^{N} \exp\left[\mathbf{k}_{m'}^T \mathbf{q}_n\right]}
\end{aligned}
$$

- Same order as before: $\mathrm{a}[\mathbf{x}_m, \mathbf{x}_n]$ is the weight the $n^{th}$ output puts on the $m^{th}$ value (UDL Eq. 12.5)



## Dot product = measure of similarity

$$
\mathbf{x}^T \mathbf{y} = |\mathbf{x}||\mathbf{y}|\cos(\theta)
$$

![](figs/10-Attention-and-Transformers/dot-product-similarity.png)

A drawback of the dot product as similarity measure is the magnitude of each vector influences the value. More rigorous to divide by magnitudes.

Cosine Similarity: $\dfrac{\mathbf{x}^T \mathbf{y}}{|\mathbf{x}||\mathbf{y}|} = \cos(\theta)$

## Motivation (revisited)

Design neural network to encode and process text:

> [The restaurant refused to serve me a ham sandwich, because it only cooks vegetarian food. In the end, they just gave me two slices of bread. Their ambience was just as good as the food and service.]{style="color: #e8890c;"}

Conclusions:

- ✓ We need a model where parameters don't increase with input length, e.g.
  $$
  \boldsymbol{\phi} = \{\boldsymbol{\beta}_v, \boldsymbol{\Omega}_v, \boldsymbol{\beta}_q, \boldsymbol{\Omega}_q, \boldsymbol{\beta}_k, \boldsymbol{\Omega}_k\}
  $$
- ✓ There must be connections between the words.
- ✓ The strength of these connections will depend on the words themselves.

## Computing Attention Weights

Ok, we defined *queries*, *keys* and *values*, but how are they used?



![a) Queries and keys from each input; b) dot products and softmax over rows give the attentions; c) the attentions as a sparse attention-weight matrix.](figs/10-Attention-and-Transformers/attention-weights-step4.png)



- Which we can then apply to the input with another sparse matrix.


The same computation in code, one output at a time, for the $N = 3$, $D = 4$
example of the routing figure. Each column of weights is non-negative and sums
to one.

In [ ]:
# Adapted from UDL notebook 12.1 (MIT)
import numpy as np

N, D = 3, 4  # number of tokens, dimension of each token
rng = np.random.default_rng(0)
all_x = [rng.normal(size=(D, 1)) for _ in range(N)]

omega_q, omega_k, omega_v = (rng.normal(size=(D, D)) for _ in range(3))
beta_q, beta_k, beta_v = (rng.normal(size=(D, 1)) for _ in range(3))


def softmax(z):
    e = np.exp(z - np.max(z))
    return e / e.sum()


queries = [beta_q + omega_q @ x for x in all_x]
keys = [beta_k + omega_k @ x for x in all_x]
values = [beta_v + omega_v @ x for x in all_x]

all_sa = []
for n in range(N):
    dots = np.array([(k_m.T @ queries[n]).item() for k_m in keys])  # k_m^T q_n
    a = softmax(dots)  # a[x_m, x_n] for m = 1..N
    all_sa.append(sum(a[m] * values[m] for m in range(N)))
    print(f"a[x_m, x_{n + 1}] = {np.round(a, 3)}   sum = {a.sum():.3f}   "
          f"sa_{n + 1} = {all_sa[n].ravel().round(3)}")

## Putting it all together...


![The value weights and the attention weights as sparse matrices.](figs/10-Attention-and-Transformers/self-attention-sparse-matrices.png)


## Putting it all together... (cont.)

Outputs =

![Outputs = attention weights × value weights × inputs.](figs/10-Attention-and-Transformers/self-attention-output-product.png)

## From Input Vector to Input Matrix

- Store all $N$ input vectors in matrix $\mathbf{X}$

![](figs/10-Attention-and-Transformers/input-matrix.png)

- Compute values, queries and keys:

$$
\begin{aligned}
\mathbf{V}[\mathbf{X}] &= \boldsymbol{\beta}_v \mathbf{1}^{\mathbf{T}} + \boldsymbol{\Omega}_{\mathbf{v}} \mathbf{X} \\
\mathbf{Q}[\mathbf{X}] &= \boldsymbol{\beta}_q \mathbf{1}^{\mathbf{T}} + \boldsymbol{\Omega}_{\mathbf{q}} \mathbf{X} \\
\mathbf{K}[\mathbf{X}] &= \boldsymbol{\beta}_k \mathbf{1}^{\mathbf{T}} + \boldsymbol{\Omega}_{\mathbf{k}} \mathbf{X},
\end{aligned}
$$

- Combine self-attentions; $\mathbf{Softmax}$ normalizes each **column** of $\mathbf{K}^T\mathbf{Q}$, one per query (the earlier figure drew rows)

$$
\mathbf{Sa}[\mathbf{X}] = \mathbf{V}[\mathbf{X}] \cdot \mathbf{Softmax}\Big[\mathbf{K}[\mathbf{X}]^T \mathbf{Q}[\mathbf{X}]\Big] = \mathbf{V} \cdot \mathbf{Softmax}[\mathbf{K}^T \mathbf{Q}]
$$

In matrix form the whole layer is four lines. $\mathbf{Softmax}$ acts on each
column (one column per query), and NumPy broadcasting supplies the
$\boldsymbol{\beta}\mathbf{1}^T$. It reproduces the loop above exactly.

In [ ]:
# Adapted from UDL notebook 12.1 (MIT)
def softmax_cols(Z):
    E = np.exp(Z - Z.max(axis=0, keepdims=True))
    return E / E.sum(axis=0, keepdims=True)


def self_attention(X, omega_v, omega_q, omega_k, beta_v, beta_q, beta_k, scale=1.0):
    V = beta_v + omega_v @ X
    Q = beta_q + omega_q @ X
    K = beta_k + omega_k @ X
    return V @ softmax_cols(K.T @ Q / scale)


X = np.hstack(all_x)  # D x N, one column per token
Sa = self_attention(X, omega_v, omega_q, omega_k, beta_v, beta_q, beta_k)

print("loop (columns sa_1..sa_N):\n", np.hstack(all_sa).round(3))
print("matrix form V Softmax[K^T Q]:\n", Sa.round(3))
print("identical:", np.allclose(Sa, np.hstack(all_sa)))

## Scaled Dot Product Self-Attention

- To avoid the case where a large value dominates the softmax $\mathbf{Softmax}[\mathbf{K}^T \mathbf{Q}]$


- you can scale the dot product by the square root of the dimension of the query

$$
\mathbf{Sa}[\mathbf{X}] = \mathbf{V} \cdot \mathbf{Softmax}\left[\frac{\mathbf{K}^T \mathbf{Q}}{\sqrt{D_q}}\right]
$$

- E.g. for $D_q = 1024$, $\sqrt{D_q} = 32$, and $x = [60, 0, -10]$

```
Unscaled softmax: [1.0 8.8e-27 4.0e-31]
Scaled softmax:   [0.79 0.12 0.089]
```


The example above, with the dimension as a variable. The second half shows
where logits like 60 come from: if the entries of $\mathbf{q}$ and $\mathbf{k}$
are independent with unit variance, $\mathbf{k}^T\mathbf{q}$ is a sum of $D_q$
terms and its standard deviation grows as $\sqrt{D_q}$. Dividing by $\sqrt{D_q}$
brings it back to 1 whatever the dimension.

In [ ]:
D_q = 1024                             # query/key dimension
logits = np.array([60.0, 0.0, -10.0])  # k_m^T q_n for one query and m = 1, 2, 3


def fmt(p):
    return "[" + ", ".join(f"{v:.4g}" for v in p) + "]"


print(f"Unscaled softmax:           {fmt(softmax(logits))}")
print(f"Scaled by 1/sqrt({D_q}) = 1/{np.sqrt(D_q):g}: {fmt(softmax(logits / np.sqrt(D_q)))}")
print()

rng = np.random.default_rng(0)
for d in (16, 64, 256, D_q):
    q = rng.normal(size=(d, 2000))
    k = rng.normal(size=(d, 2000))
    s = np.sum(q * k, axis=0)  # 2000 sample dot products k^T q
    print(f"D_q = {d:4d}: std of k^T q = {s.std():5.1f} (sqrt(D_q) = {np.sqrt(d):4.1f}),"
          f" after scaling = {(s / np.sqrt(d)).std():.2f}")

## Put it all together in matrix form


![Self-attention in matrix form, annotated.](figs/10-Attention-and-Transformers/self-attention-annotated.png)


- Values: scales linearly with sequence length, $N$
- \# attention weights scales quadratically with sequence length, $N$, but independent of length $D$ of each input
- Queries, keys and values: linear & can be calculated in parallel; attention: non-linear
- Output: linear combination of weighted inputs where weights calculated from nonlinear functions


- A *hypernetwork*: the query-key branch computes the weights that the value branch is multiplied by. The parameter count, $3D^2 + 3D$ (three weight matrices and three biases), does not depend on $N$.


## Multi-Head Self Attention

![SA outputs are vertically concatenated and combined weighted by $\boldsymbol{\Omega}_c$.](figs/10-Attention-and-Transformers/multi-head-self-attention.png)


- Multiple self-attention heads are usually applied in parallel
- $\boldsymbol{\Omega}_{qh}, \boldsymbol{\Omega}_{kh}, \boldsymbol{\Omega}_{vh}$ weight matrices would be $D/H \times D$
- "allows model to jointly attend to info from different representation subspaces at different positions"
- Original paper used 8 heads
- All can be executed in parallel


Two heads on the same toy input ($N = 3$, $D = 4$), each of width $D/H = 2$,
scaled by $1/\sqrt{D/H}$. The printout traces the shapes in the figure.

In [ ]:
# Adapted from UDL notebook 12.2 (MIT)
H = 2         # number of heads
D_h = D // H  # width of each head

rng = np.random.default_rng(1)
heads = [
    dict(omega_v=rng.normal(size=(D_h, D)), omega_q=rng.normal(size=(D_h, D)),
         omega_k=rng.normal(size=(D_h, D)), beta_v=rng.normal(size=(D_h, 1)),
         beta_q=rng.normal(size=(D_h, 1)), beta_k=rng.normal(size=(D_h, 1)))
    for _ in range(H)
]
omega_c = rng.normal(size=(D, D))


def multihead_self_attention(X, heads, omega_c):
    head_outputs = [self_attention(X, **p, scale=np.sqrt(D_h)) for p in heads]
    return omega_c @ np.vstack(head_outputs), head_outputs


MhSa, head_outputs = multihead_self_attention(X, heads, omega_c)
Q1 = heads[0]["beta_q"] + heads[0]["omega_q"] @ X
K1 = heads[0]["beta_k"] + heads[0]["omega_k"] @ X
print("input X:               ", X.shape)
print("Q_h, K_h, V_h per head:", Q1.shape)
print("attention per head:    ", (K1.T @ Q1).shape)
print("Sa_h[X] per head:      ", head_outputs[0].shape)
print("concatenated heads:    ", np.vstack(head_outputs).shape)
print("output Omega_c [...]:  ", MhSa.shape)
print("weights: 3 H (D/H) D + D^2 =", 3 * H * D_h * D + D * D, "(independent of N)")

## Equivariance to Word Order

> A function $\mathbf{f}[\mathbf{x}]$ is *equivariant* to a transformation $\mathbf{t}[\,]$ if: $\mathbf{f}[\mathbf{t}[\mathbf{x}]] = \mathbf{t}[\mathbf{f}[\mathbf{x}]]$

Self-attention is *equivariant* to permuting word order. Just a bag of words.

But word order is important in language:

[The man ate the fish]{style="color: #e8890c;"}

vs.

[The fish ate the man]{style="color: #e8890c;"}


## Solution: Position Encoding

![](figs/10-Attention-and-Transformers/transformer-positional-encoding.png)


Idea is to somehow encode *absolute* or *relative* position in the inputs

Two generations of answers:

1. 2017–2020: add a position vector to each input embedding (sinusoidal or learned)
2. 2021 onward: *rotate* the queries and keys inside every attention layer (RoPE)


## Absolute Position Encoding (2017): the sinusoidal recipe

Add a matrix $\boldsymbol{\Pi}$ to the $D \times N$ input matrix, once, before the first layer:

$$
\mathbf{X} + \boldsymbol{\Pi}
$$

Vaswani et al. pre-defined it with sinusoids of geometrically spaced frequencies; column $n$ of $\boldsymbol{\Pi}$ has entries

$$
\Pi_{2i,n} = \sin\!\left(\frac{n}{10000^{2i/D}}\right), \qquad
\Pi_{2i+1,n} = \cos\!\left(\frac{n}{10000^{2i/D}}\right)
$$

BERT and GPT-2/3 instead *learned* $\boldsymbol{\Pi}$, one $D$-vector per position up to a fixed maximum length.


![$\boldsymbol{\Pi}$ for $D = 128$.](figs/10-Attention-and-Transformers/position-encoding-matrix.png)


Some variants added $\boldsymbol{\Pi}$ to the queries, keys and values inside every layer. That form is now rare: what every layer does today is RoPE.


The matrix $\boldsymbol{\Pi}$ from the formula, and the dot product between
two of its columns. Because
$\sin a \sin b + \cos a \cos b = \cos(a - b)$, column $n$ dotted with column
$m$ is $\sum_i \cos\big((n - m)\,\omega_i\big)$: it depends only on the offset
$n - m$, so the right panel is constant along every diagonal. RoPE builds this
property into the attention score itself.

In [ ]:
#| code-fold: true
#| fig-cap: "a) The sinusoidal position-encoding matrix $\\boldsymbol{\\Pi}$ for $D = 128$ and 80 positions. b) $\\boldsymbol{\\Pi}^T\\boldsymbol{\\Pi}$: the similarity of two positions depends only on their offset."
#| fig-alt: "Two heatmaps. Left: Pi with dimension d from 0 to 127 on the vertical axis and position n from 0 to 79 on the horizontal axis; low dimensions oscillate quickly along n, high dimensions barely change, so there sine rows (near 0) and cosine rows (near 1) alternate as horizontal stripes. Right: the 80 by 80 matrix of column dot products, largest on the main diagonal and constant along each diagonal, decaying away from it."
import matplotlib.pyplot as plt

D_pe, N_pe = 128, 80  # model dimension, number of positions
n = np.arange(N_pe)
omega = 10000 ** (-2 * np.arange(D_pe // 2) / D_pe)  # one frequency per pair i

Pi = np.zeros((D_pe, N_pe))
Pi[0::2] = np.sin(np.outer(omega, n))  # rows 2i
Pi[1::2] = np.cos(np.outer(omega, n))  # rows 2i + 1

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
im = axes[0].imshow(Pi, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set(xlabel="position $n$", ylabel="dimension $d$", title=r"a) $\Pi$")
fig.colorbar(im, ax=axes[0])
im = axes[1].imshow(Pi.T @ Pi, origin="lower", cmap="viridis")
axes[1].set(xlabel="position $n$", ylabel="position $m$",
            title=r"b) $\Pi^T\Pi$: column $m$ dotted with column $n$")
fig.colorbar(im, ax=axes[1])
plt.show()

## Relative Position: what we actually want

Absolute position of a word is less important than relative position between inputs

```
          The panda eats shoots and leaves
Abs Pos:   0    1     2    3     4    5
Rel Pos:  -2   -1     0    1     2    3
```

![](figs/10-Attention-and-Transformers/attention-matrix-box.png)


Each element of the attention matrix corresponds to an offset between a query position and a key position

- One route: learn a bias $\pi_{a,b}$ per offset and add it to the dot product (Shaw et al. 2018, T5)
- A learned absolute $\boldsymbol{\Pi}$ has no entry for positions beyond the training length at all
- RoPE gets the relative dependence *for free*, with no extra parameters


## RoPE: Rotate the Queries and Keys

*Rotary Position Embedding, [Su et al. 2021](https://arxiv.org/abs/2104.09864)*

In [ ]:
#| code-fold: true
#| fig-cap: "a) One pair of a query, rotated by $m\\theta$ at position $m$; b) same offset $n - m$, same angle, same score; c) the frequency schedule."
#| fig-alt: "Three panels. a) On a unit circle, the same query vector drawn at positions m = 0, 1, 2, 3, each rotated 30 degrees further counter-clockwise. b) Two query/key pairs, q_0 with k_2 and q_3 with k_5: the angle between query and key is the same in both pairs, so the dot products are equal. c) The wavelength 2 pi over theta_i on a log axis against the dimension-pair index i from 0 to 63 for head dimension 128: a straight line rising from about 6 positions to about 60,000 for base 10,000 and to about 3 million for base 500,000, with a dashed line at context length 8,192."
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, AQUA, GRID, INK = "#2a78d6", "#eb6834", "#1baf7a", "#d9d8d4", "#52514e"
THETA = np.pi / 6  # 30 degrees per position, for a readable picture


def rotate(v, angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]])


def arrow(ax, v, color, label, alpha=1.0, offset=(0.06, 0.06)):
    ax.annotate("", xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, lw=2.2, alpha=alpha))
    ax.text(v[0] + offset[0], v[1] + offset[1], label, color=color, alpha=alpha)


def unit_circle(ax, title, formula):
    t = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), color=GRID, lw=1)
    ax.axhline(0, color=GRID, lw=0.8)
    ax.axvline(0, color=GRID, lw=0.8)
    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal")
    ax.axis("off")
    ax.set_title(title, fontsize=11, loc="left")
    ax.text(0.0, -1.15, formula, ha="center", color=INK, fontsize=12)


fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), dpi=76, constrained_layout=True)

q = np.array([1.0, 0.0])
unit_circle(axes[0], r"a) Rotate the pair by $m\theta$ at position $m$",
            r"$\tilde{\mathbf{q}}_m = \mathbf{R}_{m\theta}\,\mathbf{q}$")
for m in range(4):
    arrow(axes[0], rotate(q, m * THETA), BLUE, rf"$m={m}$", alpha=0.35 + 0.65 * m / 3)

k = rotate(np.array([0.9, 0.0]), -np.pi / 4)  # a key that is not parallel to q
unit_circle(axes[1], r"b) Same offset $n-m$, same angle, same score",
            r"$\tilde{\mathbf{q}}_0^{T}\tilde{\mathbf{k}}_2 = \tilde{\mathbf{q}}_3^{T}\tilde{\mathbf{k}}_5$")
for m, alpha in ((0, 0.45), (3, 1.0)):
    arrow(axes[1], rotate(q, m * THETA), BLUE, rf"$\tilde{{\mathbf{{q}}}}_{m}$", alpha,
          offset=(0.05, -0.16) if m == 0 else (0.06, 0.06))
    arrow(axes[1], rotate(k, (m + 2) * THETA), ORANGE, rf"$\tilde{{\mathbf{{k}}}}_{m + 2}$", alpha)

ax = axes[2]
D_head = 128
i = np.arange(D_head // 2)
for base, color, label in ((10_000, AQUA, "base 10,000 (Su et al. 2021)"),
                           (500_000, INK, "base 500,000 (Llama 3)")):
    ax.plot(i, 2 * np.pi / base ** (-2 * i / D_head), color=color, lw=2.2, label=label)
ax.axhline(8192, color=GRID, lw=1, ls="--", zorder=0)
ax.text(1, 8192 * 1.35, "context length 8,192", color=INK, fontsize=9)
ax.set(yscale="log", xlabel="Dimension pair $i$ (head dimension 128)",
       ylabel=r"Wavelength $2\pi/\theta_i$ (positions)")
ax.set_title(r"c) Frequencies $\theta_i = \mathrm{base}^{-2i/D}$", fontsize=11, loc="left")
ax.grid(True, color=GRID, lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9, loc="upper left")
plt.show()

- Take the query $\mathbf{q} = \boldsymbol{\Omega}_q \mathbf{x}_m$ for the token at position $m$ and split its $D$ entries into $D/2$ pairs $(q_{2i}, q_{2i+1})$
- Rotate pair $i$ by the angle $m\theta_i$ *as if it were a point in the plane*; do the same to the keys, with the key's own position $n$
- Nothing is added to $\mathbf{X}$ and the values $\mathbf{V}$ are untouched; it happens inside every attention layer, after the $\boldsymbol{\Omega}_q, \boldsymbol{\Omega}_k$ projections and before the dot product


## RoPE: Why the Score Depends Only on $n - m$

Work this through for one pair. The notation follows Su et al.: the query sits
at position $m$, the key at position $n$ (UDL's $\mathrm{a}[\mathbf{x}_m, \mathbf{x}_n]$
indexes the other way round, with $n$ for the output).


A rotation by angle $\alpha$ in the plane is $\mathbf{R}_\alpha = \begin{pmatrix} \cos\alpha & -\sin\alpha \\ \sin\alpha & \cos\alpha \end{pmatrix}$, and rotations compose by adding angles: $\mathbf{R}_\alpha^T = \mathbf{R}_{-\alpha}$ and $\mathbf{R}_{-\alpha}\mathbf{R}_\beta = \mathbf{R}_{\beta - \alpha}$.

RoPE sets $\tilde{\mathbf{q}}_m = \mathbf{R}_{m\theta}\,\mathbf{q}$ and $\tilde{\mathbf{k}}_n = \mathbf{R}_{n\theta}\,\mathbf{k}$, so the attention logit is

$$
\tilde{\mathbf{q}}_m^T \tilde{\mathbf{k}}_n
= \mathbf{q}^T \mathbf{R}_{m\theta}^T \mathbf{R}_{n\theta}\, \mathbf{k}
= \mathbf{q}^T \mathbf{R}_{(n-m)\theta}\, \mathbf{k}
$$


Written out with $\mathbf{q} = (q_1, q_2)$, $\mathbf{k} = (k_1, k_2)$ and $\alpha = (n - m)\theta$:

$$
\tilde{\mathbf{q}}_m^T \tilde{\mathbf{k}}_n = (q_1 k_1 + q_2 k_2)\cos\alpha + (q_2 k_1 - q_1 k_2)\sin\alpha
$$

The absolute positions $m$ and $n$ have cancelled: the score is the *content* dot product, rotated by the *offset* only. Shift the whole sentence by 100 tokens and every attention weight is unchanged.


A check you can do by hand: take $\theta = 90^\circ$, $\mathbf{q} = (1, 0)$ and
$\mathbf{k} = (1, 0)$. With the query at $m = 0$ and the key at $n = 1$ the
logit is $\cos 90^\circ = 0$; with $m = 7$ and $n = 8$ it is again $0$; with
$m = n$ it is $1$. Shifting both positions by the same amount never changes the
answer, and the sign of the $\sin\alpha$ term tells the model whether the key
is to the left or the right of the query.

The same check numerically, for a full 64-dimensional head with the frequency
schedule of the next slide. `rope` is the "two elementwise multiplies against
cached $\cos$ and $\sin$ tables" form. Every pair with offset $n - m = 2$ gives
the same score, however far down the sequence it sits; other offsets do not.

In [ ]:
def rope(x, pos, base=10_000):
    """Rotate each pair (x[2i], x[2i+1]) of a D-vector by the angle pos * theta_i."""
    theta = base ** (-2 * np.arange(len(x) // 2) / len(x))
    cos, sin = np.cos(pos * theta), np.sin(pos * theta)
    out = np.empty_like(x)
    out[0::2] = x[0::2] * cos - x[1::2] * sin
    out[1::2] = x[0::2] * sin + x[1::2] * cos
    return out


rng = np.random.default_rng(0)
D_head = 64
q, k = rng.normal(size=D_head), rng.normal(size=D_head)

for m, n in [(0, 2), (3, 5), (10, 12), (100, 102), (5000, 5002), (0, 0), (0, 1), (0, 7)]:
    print(f"m = {m:4d}, n = {n:4d}, n - m = {n - m}:  "
          f"rope(q, m)^T rope(k, n) = {rope(q, m) @ rope(k, n):+.6f}")
print(f"unrotated q^T k = {q @ k:+.6f} (equals the n - m = 0 score)")

## RoPE: the Frequency Schedule

Each pair $i = 0, \ldots, D/2 - 1$ has its own angular frequency, the same geometric schedule as the 2017 sinusoids:

$$
\theta_i = b^{-2i/D}, \qquad b = 10{,}000 \text{ in Su et al.}
$$

so the full rotation is block-diagonal, $\mathbf{R}_m = \mathrm{diag}(\mathbf{R}_{m\theta_0}, \ldots, \mathbf{R}_{m\theta_{D/2-1}})$, and $D$ here is the per-head dimension (64 or 128), not the model width.


- Pair 0 completes a turn every $2\pi \approx 6$ tokens: it resolves *adjacent* words
- The last pair's wavelength is nearly $2\pi b$, over 50,000 tokens: it encodes *coarse* distance
- Raising the base $b$ stretches every wavelength (panel c); Llama 3 uses $b = 500{,}000$ ([Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783))
- No parameters; in code it is two elementwise multiplies against cached $\cos$ and $\sin$ tables


## Extending the Context: NTK Scaling and YaRN

A model trained on $L$ positions is asked for $L' = sL$ at inference. The low-frequency pairs now reach angles they have never seen, and the loss blows up.

- **Position interpolation** ([Chen et al. 2023](https://arxiv.org/abs/2306.15595)): use $m/s$ instead of $m$, so every angle stays inside the trained range. Needs a short fine-tune, and it crowds together nearby tokens, which the high-frequency pairs were resolving.
- **NTK-aware scaling** (bloc97, 2023): raise the base $b$ instead of dividing $m$. The low-frequency pairs get interpolated, the high-frequency pairs barely change, so local resolution survives.
- **YaRN** ([Peng et al. 2023](https://arxiv.org/abs/2309.00071)): do it per pair. Leave pairs whose wavelength is much shorter than $L$ alone, interpolate those whose wavelength exceeds $L$, ramp in between, and sharpen the softmax by scaling the logits by $(0.1 \ln s + 1)^2$ (YaRN eq. 15), i.e. a temperature that shrinks as $s$ grows. Extends Llama 2 from 4k to 128k by fine-tuning on less than about 0.1% of the pretraining data.


**Why RoPE made this possible.** Position enters only through the angle table $\{\theta_i\}$: there is nothing to learn, so changing the table is a change of *inference-time hyperparameters*, not of weights, and the relative-offset property stays exact. A learned $\boldsymbol{\Pi}$ has no row for position $L + 1$. Lecture 11 shows `rope_theta` and the scaling block in a real open-weight config.


## Visualization with BertViz

![](figs/10-Attention-and-Transformers/bertviz-attention.png)


- 12 heads


<https://github.com/jessevig/bertviz>, [Colab Demo](https://colab.research.google.com/drive/1hXIQ77A4TYS4y3UthWF-Ci7V7vVUoxmQ?usp=sharing)

# The Transformer Architecture

Multi-head self-attention is one component of a transformer layer, which also
adds an MLP, residual connections and LayerNorm.


## Transformers


![](figs/10-Attention-and-Transformers/transformer-encoder-block.png)


- *Multi-headed Self Attention* is just one component of the transformer architecture

- Let's look at a transformer *block* (or *layer*) from the encoder


## Transformer Layer -- Complete

![](figs/10-Attention-and-Transformers/transformer-layer.png)

- Adds a 2-layer MLP
- Adds residual connections around multi-head self-attentions and the parallel MLPs
- Adds LayerNorm, which normalizes each token $\mathbf{x}_n$ over its $D$ features (not across the $N$ tokens)


**Transformer Layer**

$$
\begin{aligned}
\mathbf{X} &\leftarrow \mathbf{X} + \mathbf{MhSa}[\mathbf{X}] \\
\mathbf{X} &\leftarrow \mathbf{LayerNorm}[\mathbf{X}] \\
\mathbf{x}_n &\leftarrow \mathbf{x}_n + \mathbf{mlp}[\mathbf{x}_n] \\
\mathbf{X} &\leftarrow \mathbf{LayerNorm}[\mathbf{X}],
\end{aligned}
$$


## Transformer Layer -- MLP


- Adds 2-layer MLP
    - Same network (same weights) operates independently on each word
    - Learn more complex representations and expand model capacity

$$
\mathrm{Linear}_{D \times 4D} \rightarrow \mathrm{ReLU}(.) \rightarrow \mathrm{Linear}_{4D \times D}
$$

## Transformer Layer -- LayerNorm

![](figs/10-Attention-and-Transformers/transformer-layer-layernorm.png)

- Normalizes each token over its $D$ features, independently of the other tokens and of the batch
- Learned gain $\gamma_i$ and offset $\beta_i$ per feature

For $\mathbf{x}_n \in \mathbb{R}^D$:

$$
\mu_n = \frac{1}{D}\sum_{i=1}^{D} x_{n,i}, \qquad \sigma_n^2 = \frac{1}{D}\sum_{i=1}^{D}\left(x_{n,i} - \mu_n\right)^2
$$

$$
\mathrm{LayerNorm}(x_{n,i}) = \gamma_i \frac{x_{n,i} - \mu_n}{\sqrt{\sigma_n^2 + \epsilon}} + \beta_i
$$


```python
# NLP Example
batch, sentence_length, embedding_dim = 20, 5, 10
embedding = torch.randn(batch, sentence_length, embedding_dim)
layer_norm = nn.LayerNorm(embedding_dim)

# Activate module
layer_norm(embedding)
```

<https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html>


# Three Types of NLP Transformer Models

Transformer layers are assembled into three kinds of language model: encoders,
decoders and encoder-decoders.


## 3 Types of Transformer Models

1. *Encoder* -- transforms text embeddings into representations that support variety of tasks (e.g. sentiment analysis, classification)
    - Model Example: BERT
2. *Decoder* -- predicts the next token to continue the input text (e.g. ChatGPT, AI assistants)
    - Model Examples: GPT-3 and nearly every 2026 LLM (Llama, Qwen, Gemma, DeepSeek)
3. *Encoder-Decoder* -- used in sequence-to-sequence tasks, where one text string is converted to another (e.g. machine translation)

# Encoder

## Encoder Model Example: BERT (2019)

*<u>B</u>idirectional <u>E</u>ncoder <u>R</u>epresentations from <u>T</u>ransformers*

- Hyperparameters
    - 30,000 token vocabulary
    - 1024-dimensional word embeddings
    - 24x transformer layers
    - 16 heads in self-attention mechanism
    - 4096 hidden units in middle of MLP
- ~340 million parameters
- *Pre-trained* in a *self-supervised* manner,
- then can be adapted to task with one additional layer and *fine-tuned*

J. Devlin, M.-W. Chang, K. Lee, and K. Toutanova, "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding." arXiv, May 24, 2019. doi: [10.48550/arXiv.1810.04805](https://doi.org/10.48550/arXiv.1810.04805).

## Encoder Pre-Training

![Special `<cls>` token used for aggregate sequence representation for classification.](figs/10-Attention-and-Transformers/bert-pretraining.png)

- A small percentage of input embedding replaced with a generic `<mask>` token
- Predict missing token from output embeddings
- Added linear layer and softmax to generate probabilities over vocabulary
- Trained on BooksCorpus (800M words) and English Wikipedia (2,500M words)

## Encoder Fine-Tuning

![**a)** Sentiment Analysis, using the `<cls>` token position. **b)** Named Entity Recognition (NER).](figs/10-Attention-and-Transformers/bert-finetuning.png)


- Extra layer(s) appended to convert output vectors to desired output format
- 3^rd^ Example: Text span prediction -- predict start and end location of answer to a question in passage of Wikipedia, see [SQuAD explorer](https://rajpurkar.github.io/SQuAD-explorer/)


# Decoder

## Decoder Model Example: GPT3 (2020)

*<u>G</u>enerative <u>P</u>re-trained <u>T</u>ransformer*

- One purpose: *generate the next token in a sequence*
- By constructing an autoregressive model

- Factors the probability of the sentence:

$$
\begin{aligned}
\mathrm{Pr}(\textit{Learning deep learning is fun}) = \;& \mathrm{Pr}(\textit{Learning}) \times \mathrm{Pr}(\textit{deep} \mid \textit{Learning}) \times \\
& \mathrm{Pr}(\textit{learning} \mid \textit{Learning deep}) \times \\
& \mathrm{Pr}(\textit{is} \mid \textit{Learning deep learning}) \times \\
& \mathrm{Pr}(\textit{fun} \mid \textit{Learning deep learning is})
\end{aligned}
$$


- More formally: Autoregressive model

$$
\mathrm{Pr}(t_1, t_2, \ldots, t_N) = \mathrm{Pr}(t_1) \prod_{n=2}^{N} \mathrm{Pr}(t_n \mid t_1, t_2, \ldots, t_{n-1})
$$


T. B. Brown *et al.*, "Language Models are Few-Shot Learners." arXiv, Jul. 22, 2020. doi: [10.48550/arXiv.2005.14165](https://doi.org/10.48550/arXiv.2005.14165).


## Decoder: *Masked* Self-Attention

![](figs/10-Attention-and-Transformers/decoder-masked-attention.png)

- During training we want to maximize the log probability of the input text under the autoregressive model
- We want to make sure the model doesn't "cheat" during training by looking ahead at the next token
- Hence we set the dot products for the *right* context (later tokens) to *negative infinity* before the softmax, so those weights are zero; each token still attends to itself and to its left

## Masked Self-Attention

![Mask right context self-attention weights to zero.](figs/10-Attention-and-Transformers/masked-attention-routing.png)

## Masked Self-Attention (cont.)

![](figs/10-Attention-and-Transformers/masked-attention-matrix.png)

Causal masking for $N = 5$ tokens. In the $\mathbf{K}^T\mathbf{Q}$ layout, row
$m$ is a key and column $n$ a query, so the masked entries (key after query,
$m > n$) are *below* the diagonal; the figure above draws the transpose, one
row per query. After the column softmax, token 1 attends only to itself and
token 5 to everything. The last lines check that the model cannot "cheat":
changing the final token leaves every earlier output untouched.

In [ ]:
def causal_self_attention(X, omega_v, omega_q, omega_k, beta_v, beta_q, beta_k):
    V = beta_v + omega_v @ X
    Q = beta_q + omega_q @ X
    K = beta_k + omega_k @ X
    scores = K.T @ Q / np.sqrt(Q.shape[0])
    m, n = np.indices(scores.shape)
    scores[m > n] = -np.inf  # key m comes after query n: hide it
    A = softmax_cols(scores)
    return V @ A, A


N_c = 5
rng = np.random.default_rng(0)
X5 = rng.normal(size=(D, N_c))
params = (omega_v, omega_q, omega_k, beta_v, beta_q, beta_k)

out, A = causal_self_attention(X5, *params)
print("attention a[x_m, x_n] (row m = key, column n = query):")
print(A.round(3))
print("column sums:", A.sum(axis=0).round(3))

X5_changed = X5.copy()
X5_changed[:, -1] = rng.normal(size=D)  # replace the last token
out_changed, _ = causal_self_attention(X5_changed, *params)
print("outputs 1-4 unchanged:", np.allclose(out[:, :-1], out_changed[:, :-1]))
print("output 5 unchanged:   ", np.allclose(out[:, -1], out_changed[:, -1]))

## Decoder: Text Generation (Generative AI)


![Text generation: the prompt "\<start\> It takes great" is fed in, outputs at the prompt positions are ignored, and each generated token is fed back into the input.](figs/10-Attention-and-Transformers/text-generation-step3.png)


- Prompt with token string "`<start>` It takes great"
- Generate next token for the sequence by
    - picking most likely token
    - sample from the probability distribution
        - alternative *top-k* sampling to avoid picking from the long tail
    - beam search -- select the most likely sentence rather than greedily pick

- Feed the output back into input


## Technical Details

|                    | BERT        | GPT3         |
|--------------------|-------------|--------------|
| Model Architecture | Encoder     | Decoder      |
| Embedding Size     | 1024        | 12,288       |
| Vocabulary         | 30K tokens  |              |
| Sequence Length    |             | 2048         |
| # Heads            | 16          | 96           |
| # Layers           | 24          | 96           |
| Q,K,V dimensions   | 64          | 128          |
| Training set size  | 3.3B words  | 300B tokens  |
| # Parameters       | 340M        | 175B         |

# Encoder-Decoder

## Encoder-Decoder Model

- Used for *machine translation*, which is a *sequence-to-sequence* task

![Figure: Jay Alammar, [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/).](figs/10-Attention-and-Transformers/encoder-decoder-translation.png)

## Encoder Decoder Model (cont.)

![](figs/10-Attention-and-Transformers/transformer-cross-attention-stage.png)


- The transformer layer in the decoder of the encoder-decoder model has an extra stage
- Attends to the input of the encoder with *cross attention* using Keys and Values from the output of the encoder
- Shown here on original diagram from "Attention is all you need" paper


## Encoder Decoder Model (UDL view)

![](figs/10-Attention-and-Transformers/udl-encoder-decoder.png)


- Same view per UDL book


## Cross-Attention

![Keys and Values come from the last stage of the encoder.](figs/10-Attention-and-Transformers/cross-attention.png)

# Summary

One line per idea; the recap deck uses these.


## Summary

- A fully connected layer over $N$ tokens needs $(DN)^2$ weights and a fixed $N$; self-attention needs $3D^2 + 3D$ and any $N$.
- Each output is a weighted sum of values, $\mathbf{sa}_n = \sum_m \mathrm{a}[\mathbf{x}_m, \mathbf{x}_n]\,\mathbf{v}_m$, with weights $\mathrm{a}[\mathbf{x}_m, \mathbf{x}_n] = \mathrm{softmax}_m[\mathbf{k}_m^T \mathbf{q}_n]$ computed from the inputs themselves.
- Matrix form $\mathbf{Sa}[\mathbf{X}] = \mathbf{V}\,\mathbf{Softmax}[\mathbf{K}^T\mathbf{Q}/\sqrt{D_q}]$: the $N \times N$ attention is quadratic in $N$, the values linear; scaling by $\sqrt{D_q}$ keeps the softmax from saturating.
- $H$ heads of width $D/H$ run in parallel and are concatenated and mixed by $\boldsymbol{\Omega}_c$.
- Self-attention is permutation-equivariant, so position must be injected: 2017 added sinusoidal or learned $\boldsymbol{\Pi}$ to the input; today RoPE rotates each query/key pair by $m\theta_i$ with $\theta_i = b^{-2i/D}$.
- RoPE makes $\tilde{\mathbf{q}}_m^T \tilde{\mathbf{k}}_n = \mathbf{q}^T \mathbf{R}_{(n-m)\theta}\,\mathbf{k}$, a function of the offset only; context extension (NTK scaling, YaRN) just rescales the angle table.
- A transformer layer is $\mathbf{X} \leftarrow \mathbf{X} + \mathbf{MhSa}[\mathbf{X}]$, LayerNorm, per-token MLP with a residual, LayerNorm; LayerNorm normalizes each token over its $D$ features.
- Encoders (BERT) see both directions and are pre-trained by masking; decoders (GPT) mask the right context and are trained autoregressively; encoder-decoders add cross-attention with keys and values from the encoder.
- Generation feeds each sampled token back in, one at a time.

Lecture 09 covered the recurrent alternatives that attention replaced and the
linear-time recurrences that are now mixed back in. Lecture 11 looks at what the
2026 decoder actually looks like: grouped-query attention, mixture of experts,
SwiGLU and RoPE as it appears in a real model config.
